# mini-vLLM — GPU Benchmark Runner (Colab)

Runs the benchmark scripts (RoPE, Flash Attention, INT8 quantization, chunked
prefill, **CUDA-graph decode**) **and** the test suite on a Colab GPU, then
writes a clean `docs/benchmarks.md` (GPU name + date + tables).

### Before you run
1. **Push the code to GitHub first.** This notebook *clones* the repo, so the
   Triton kernels, quantization, chunked-prefill scheduler, CUDA-graph runner,
   and the `scripts/benchmark_*.py` files must already be on the branch you set
   as `BRANCH` below. Cell 4 prints `MISSING` for any file that isn't there.
2. **Runtime → Change runtime type → GPU** (a free **T4** is plenty).

Colab already ships a matched `torch` + `triton`, so we install only the
model/runtime dependencies — we deliberately do **not** reinstall torch.

When it finishes, download `docs/benchmarks.md` (last cell) or copy the printed
contents back.

In [ ]:
# 1. GPU + toolchain check
!nvidia-smi
import torch, triton
print("torch  :", torch.__version__)
print("triton :", triton.__version__)
print("CUDA   :", torch.cuda.is_available())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime to GPU!")

In [ ]:
# 2. Install model/runtime deps (torch + triton already present on Colab)
!pip install -q transformers huggingface_hub safetensors pytest \
    fastapi "uvicorn[standard]" pydantic prometheus-client httpx websockets
print("deps installed")

In [ ]:
# 3. Clone the repo (or hard-sync it to the latest pushed commit).
import os, subprocess
REPO   = "https://github.com/vsvidhun06-blip/mini-vllm.git"
BRANCH = "main"          # <-- set to the branch that has the code pushed

if not os.path.isdir("mini-vllm"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO], check=True)
else:
    # The clone PERSISTS across cell re-runs on the same Colab runtime, so a
    # plain "clone only if missing" keeps serving STALE code — the #1 reason a
    # pushed fix appears not to "reach Colab". Force the working tree to exactly
    # match the latest pushed commit on BRANCH.
    subprocess.run(["git", "-C", "mini-vllm", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", "mini-vllm", "reset", "--hard", f"origin/{BRANCH}"], check=True)
REPO_ROOT = os.path.abspath("mini-vllm")

# Belt-and-suspenders: delete any stale bytecode so Python can't import an old
# cached .pyc (or a leftover __pycache__) instead of the freshly-synced source.
!find /content/mini-vllm -name "*.pyc" -delete
!find /content/mini-vllm -name "__pycache__" -type d -exec rm -rf {} +

print("repo root:", REPO_ROOT)
print(subprocess.run(["git", "-C", REPO_ROOT, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

# Sanity: do the expected files exist on this branch? If any say MISSING,
# the code wasn't pushed — fix that and re-run this cell.
need = [
    "scripts/benchmark_rope.py",
    "scripts/benchmark_flash_attention.py",
    "scripts/benchmark_quantization.py",
    "scripts/benchmark_chunked_prefill.py",
    "scripts/benchmark_cuda_graphs.py",
    "scripts/benchmark_kv_eviction.py",
    "src/engine/kernels/rope.py",
    "src/engine/kernels/flash_attention.py",
    "src/engine/kernels/quantization.py",
    "src/engine/kernels/quant_attention.py",
    "src/engine/cuda_graph.py",
    "src/engine/kv_eviction.py",
]
for f in need:
    print(("OK      " if os.path.exists(os.path.join(REPO_ROOT, f)) else "MISSING "), f)

In [ ]:
# 4. Pre-download TinyLlama weights + tokenizer (so the model benchmarks and
#    pytest don't each re-fetch, and a kernel issue can't block the download).
env = {**os.environ, "PYTHONPATH": REPO_ROOT}
pre = (
    "from src.engine.model import MODEL_NAME, load_tinyllama_from_hf\n"
    "from transformers import AutoTokenizer\n"
    "AutoTokenizer.from_pretrained(MODEL_NAME)\n"
    "load_tinyllama_from_hf(MODEL_NAME)\n"
    "print('model + tokenizer cached')\n"
)
p = subprocess.run(["python", "-c", pre], cwd=REPO_ROOT, env=env,
                   capture_output=True, text=True)
print(p.stdout[-1500:])
if p.returncode != 0:
    print("[stderr]\n", p.stderr[-3000:])

In [ ]:
# 5. Helpers: run a script capturing output, and turn a pipe-delimited table
#    into a GitHub-markdown table.
def run(cmd):
    p = subprocess.run(cmd, cwd=REPO_ROOT, env=env, capture_output=True, text=True)
    print("$", " ".join(cmd))
    print(p.stdout)
    if p.returncode != 0:
        print("[exit %d] stderr:\n%s" % (p.returncode, p.stderr[-4000:]))
    return {"cmd": " ".join(cmd), "returncode": p.returncode,
            "stdout": p.stdout, "stderr": p.stderr}

def to_md_table(raw):
    lines = [l for l in raw.splitlines() if "|" in l]
    def is_sep(l):
        return l.replace("|", "").replace("-", "").replace("+", "").strip() == ""
    rows = [l for l in lines if not is_sep(l)]
    if not rows:
        return None
    def cells(l):
        return [c.strip() for c in l.strip().strip("|").split("|")]
    header = cells(rows[0])
    out = ["| " + " | ".join(header) + " |",
           "| " + " | ".join("---" for _ in header) + " |"]
    for r in rows[1:]:
        out.append("| " + " | ".join(cells(r)) + " |")
    return "\n".join(out)

In [ ]:
# 6. Run the benchmarks (order: cheap tensor benches first, model benches last)
results = {}
results["RoPE kernel — PyTorch vs Triton"]        = run(["python", "scripts/benchmark_rope.py"])
results["Flash Attention — SDPA vs Triton FA2"]   = run(["python", "scripts/benchmark_flash_attention.py"])
results["INT8 quantization — fp32 vs W8A8"]       = run(["python", "scripts/benchmark_quantization.py"])
results["Chunked prefill — full vs chunked"]      = run(["python", "scripts/benchmark_chunked_prefill.py"])
results["CUDA graphs — eager vs graph decode"]    = run(["python", "scripts/benchmark_cuda_graphs.py"])
results["KV eviction (H2O) — long context"]       = run(["python", "scripts/benchmark_kv_eviction.py"])
results["Disaggregated prefill/decode — unified vs split"] = run(["python", "scripts/benchmark_disaggregated.py"])


In [ ]:
# 6b. LLM Router — classify + route 1000 synthetic prompts.
#     This one is CPU-only and loads no model (the router decides where to send
#     a request without paying a model's worth of compute), so it runs fast and
#     would pass even without a GPU. It reports the traffic split across the
#     small/large fleet, the cost saved vs always using the large model, and the
#     per-request routing overhead (P50/P95/P99). Folded into `results` so it
#     lands in docs/benchmarks.md alongside the kernel benches.
results["LLM Router — 1000-prompt routing + cost savings"] = run(
    ["python", "scripts/benchmark_router.py"]
)

In [ ]:
# 6d. CARL evaluation suite — ablation study (FAST PREVIEW).
#     This is the rigorous-evaluation counterpart to cell 6c. Where 6c MEASURES
#     real TinyLlama inference, this drives a torch-free CONTROL-LOOP SIMULATION:
#     it runs the real CARL controller/bandits/AutoTuner over an analytical cost
#     model (scripts/benchmark_carl.py's WorkloadModel) and ablates each adaptive
#     subsystem in turn. COMPARISONS between configs are the result, not absolute
#     tok/s, and the regime oracle is near-optimal BY CONSTRUCTION — see
#     docs/eval/README.md for the full caveats and the other four eval scripts
#     (workload_suite, oracle_comparison, sensitivity, statistical_validation).
#
#     Reduced settings here (3 runs x 20 requests) so the Colab preview is quick;
#     the full suite is CPU-only and finishes in seconds when run locally:
#         python scripts/eval/ablation.py            # 5 runs x 30 requests
#     The table (CARL-Full vs the 5 ablations vs Static-Best/AutoTuner/Oracle,
#     plus a per-subsystem contribution summary) lands in docs/benchmarks.md.
results["CARL eval — ablation study (simulation, fast preview)"] = run(
    ["python", "scripts/eval/ablation.py", "--runs", "3", "--requests", "20"]
)


In [ ]:
# 7. Run the full test suite
pytest_res = run(["python", "-m", "pytest", "tests/", "-v"])

In [ ]:
# 8. Assemble docs/benchmarks.md
import datetime
gpu = torch.cuda.get_device_name(0)
parts = [
    "# mini-vLLM — GPU Benchmarks\n",
    f"- **GPU:** {gpu}",
    f"- **Date:** {datetime.date.today().isoformat()}",
    f"- **torch:** {torch.__version__} · **triton:** {triton.__version__}",
    "",
    "_Generated on Google Colab by `docs/run_benchmarks.ipynb`._",
    "",
]
for title, res in results.items():
    parts.append(f"## {title}\n")
    if res["returncode"] != 0:
        parts.append(f"> ⚠️ Script exited with code {res['returncode']} — see raw output below.\n")
    tbl = to_md_table(res["stdout"])
    if tbl:
        parts.append(tbl + "\n")
    body = res["stdout"].strip()
    if res["stderr"].strip():
        body += "\n\n--- stderr ---\n" + res["stderr"].strip()
    parts.append("<details><summary>Raw output</summary>\n\n```\n" + body + "\n```\n</details>\n")

# Test suite
parts.append("## Test suite — `pytest tests/ -v`\n")
summary = [l for l in pytest_res["stdout"].splitlines()
           if any(k in l for k in (" passed", " failed", " error", " skipped"))]
if summary:
    parts.append("```\n" + summary[-1].strip() + "\n```\n")
ptbody = pytest_res["stdout"].strip()
if pytest_res["stderr"].strip():
    ptbody += "\n\n--- stderr ---\n" + pytest_res["stderr"].strip()
parts.append("<details><summary>Full pytest output</summary>\n\n```\n" + ptbody + "\n```\n</details>\n")

md = "\n".join(parts) + "\n"
os.makedirs(os.path.join(REPO_ROOT, "docs"), exist_ok=True)
out_path = os.path.join(REPO_ROOT, "docs", "benchmarks.md")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(md)
print("wrote", out_path, f"({len(md)} chars)\n")
print("=" * 72)
print(md)

In [ ]:
# 9. Download docs/benchmarks.md (or just copy the printed text above back).
from google.colab import files
files.download(os.path.join(REPO_ROOT, "docs", "benchmarks.md"))